# Image Viewer — демонстрация работы через командную строку

Блокнот показывает основные сценарии проекта **«Инструмент полноэкранного просмотра изображений»**:

1. сборка решения и запуск тестов;
2. консольная утилита `ivc` — она использует **ту же библиотеку `ImageViewer.Core`**, что и WPF-приложение;
3. запуск графического просмотрщика `ImageViewer.exe` с параметрами командной строки.

**Требования:** .NET SDK 10, Python 3 с Jupyter (`pip install notebook matplotlib`). Ядро блокнота — Python.
Запускайте ячейки по порядку сверху вниз. Рабочая папка блокнота — `demo/`.

In [ ]:
import json, os, platform, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
IMAGES = ROOT / "demo" / "images"
IS_WINDOWS = platform.system() == "Windows"
CLI_PROJECT = ROOT / "src" / "ImageViewer.Cli"

def run(*args, check=True):
    """Запускает команду и печатает её вывод."""
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run([str(a) for a in args], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        print(f"[код возврата {result.returncode}]")
    return result

def ivc(*args):
    """Консольная утилита ivc (через dotnet run, без повторной сборки)."""
    return run("dotnet", "run", "--project", CLI_PROJECT, "-c", "Release", "--no-build", "--", *args, check=False)

print("Корень решения:", ROOT)
print("ОС:", platform.system())

## 1. Сборка решения и автоматические тесты

In [ ]:
# Windows — всё решение (WPF + Avalonia); Linux/macOS — фильтр без Windows-проектов (Avalonia, консоль, тесты)
if IS_WINDOWS:
    run("dotnet", "build", "ImageViewer.slnx", "-c", "Release")
else:
    run("dotnet", "build", "ImageViewer.Linux.slnf", "-c", "Release")

In [ ]:
run("dotnet", "test", "tests/ImageViewer.Core.Tests", "-c", "Release", "--logger", "console;verbosity=normal")

## 2. Консольная утилита `ivc`

### Справка и список команд

In [ ]:
ivc("help")

### Тестовый набор изображений
Команда `generate` создаёт BMP-файлы с разными узорами. Имена `demo_1 … demo_10` позволяют проверить
**естественную сортировку** (demo_2 идёт раньше demo_10). Среди них есть «панорама» (широкое) и «длинный скриншот» (высокое).

In [ ]:
ivc("generate", IMAGES, "--count", "10", "--size", "800x600")

### Информация о файле: размер, тип, формат, разрешение, глубина цвета

In [ ]:
ivc("info", IMAGES / "demo_1.bmp")

In [ ]:
# Тот же результат в JSON — удобно для обработки в Python
info = json.loads(ivc("info", IMAGES / "demo_5.bmp", "--json").stdout)
print(f"{info['fileName']}: {info['width']}×{info['height']}, {info['fileSizeText']}, {info['formatName']}")

### Список изображений папки — порядок навигации, номер текущего и общее количество

In [ ]:
ivc("list", IMAGES / "demo_3.bmp")

In [ ]:
ivc("list", IMAGES, "--sort", "size", "--desc")

### Навигация по папке: следующее, предыдущее, первое, последнее, по кругу

In [ ]:
ivc("nav", IMAGES / "demo_9.bmp", "--step", "1")          # следующее
ivc("nav", IMAGES / "demo_10.bmp", "--step", "1")         # после последнего — снова первое
ivc("nav", IMAGES / "demo_10.bmp", "--step", "1", "--no-wrap")  # без перехода по кругу
ivc("nav", IMAGES / "demo_4.bmp", "--step", "-3")         # назад на 3
ivc("nav", IMAGES / "demo_4.bmp", "--last")

### Режимы масштаба: 100%, вписать, по ширине, по высоте, заполнить
Расчёт выполняет класс `ZoomController` — тот же, что управляет картинкой в WPF-окне.

In [ ]:
ivc("fit", "4000x3000", "--viewport", "1920x1080")

In [ ]:
ivc("fit", IMAGES / "demo_5.bmp", "--viewport", "1280x720")   # широкая «панорама»

In [ ]:
ivc("fit", IMAGES / "demo_7.bmp", "--viewport", "1280x720", "--rotate", "90")  # высокое + поворот

### Гистограмма

In [ ]:
ivc("histogram", IMAGES / "demo_1.bmp", "--channel", "all", "--width", "64", "--height", "8")

In [ ]:
# Строим гистограмму в matplotlib по JSON-данным из ivc
import matplotlib.pyplot as plt

data = json.loads(ivc("histogram", IMAGES / "demo_4.bmp", "--json").stdout)
fig, ax = plt.subplots(figsize=(9, 3.5))
for key, color in [("red", "#e04040"), ("green", "#30a030"), ("blue", "#3070e0")]:
    ax.fill_between(range(256), data[key], alpha=0.35, color=color, label=key)
ax.plot(range(256), data["luminance"], color="black", linewidth=1, label="luminance")
ax.set_title(f"{data['file']} — {data['width']}×{data['height']}, средняя яркость {data['meanLuminance']}")
ax.set_xlim(0, 255)
ax.legend()
plt.show()

### Параметры командной строки графического приложения
`check-args` разбирает параметры **тем же парсером `CommandLineParser`**, что и `ImageViewer.exe`, но не открывает окно.
Всё, что после `--`, передаётся парсеру как есть.

In [ ]:
ivc("check-args", "--", "photo.jpg", "--zoom", "width", "--fullscreen", "-t", "light", "-b", "checker", "--slideshow", "4")
ivc("check-args", "--", "C:\\Photos", "--sort", "date", "--desc", "--no-wrap", "-n", "3", "--info")
ivc("check-args", "--", "a.jpg", "--zoom", "huge", "--bogus")   # ошибки

### Горячие клавиши и пользовательские настройки

In [ ]:
ivc("keys")

In [ ]:
ivc("settings")

## 3. Графический просмотрщик

Два интерфейса к одной библиотеке:
* **Avalonia** (`ImageViewerX`) — Linux, Windows, macOS;
* **WPF** (`ImageViewer.exe`) — только Windows.

Каждая ячейка открывает окно — закройте его клавишей `Esc` (в полноэкранном режиме первое нажатие
возвращает в оконный режим) или `Ctrl+Q`. В окне нажмите **F1** — справка.

In [ ]:
AVALONIA = ROOT / "src" / "ImageViewer.Avalonia" / "bin" / "Release" / "net10.0" / ("ImageViewerX.exe" if IS_WINDOWS else "ImageViewerX")
WPF = ROOT / "src" / "ImageViewer.Wpf" / "bin" / "Release" / "net10.0-windows" / "ImageViewer.exe"

# На Linux — Avalonia; на Windows можно выбрать: USE_WPF = True — WPF-версия, False — Avalonia
USE_WPF = IS_WINDOWS
GUI = WPF if USE_WPF else AVALONIA

def gui(*args):
    if not GUI.exists():
        print("Не найдено:", GUI, "— соберите решение (раздел 1)")
        return
    print(">", GUI.name, *args)
    subprocess.Popen([str(GUI), *[str(a) for a in args]], cwd=ROOT)

print("Графический интерфейс:", "WPF" if USE_WPF else "Avalonia", "—", GUI)

In [ ]:
# Открыть файл на весь экран с режимом «вписать»
gui(IMAGES / "demo_1.bmp", "--zoom", "fit", "--fullscreen")

In [ ]:
# 100%, панель информации с гистограммой, в окне
gui(IMAGES / "demo_4.bmp", "--zoom", "100", "--info", "--windowed")

In [ ]:
# Высокое изображение «по ширине», светлая тема, шахматный фон
gui(IMAGES / "demo_7.bmp", "-z", "width", "-t", "light", "-b", "checker", "-w")

In [ ]:
# Слайд-шоу по папке: сортировка по размеру, интервал 2 секунды, начать с 3-го изображения
gui(IMAGES, "--sort", "size", "--slideshow", "2", "-n", "3")

In [ ]:
# Справка по параметрам командной строки
gui("--help")

## Итог

| Требование | Где показано |
|---|---|
| Файл через командную строку, выбор другого по горячей клавише | раздел 3, `Ctrl+O` в окне |
| Зумирование, перемещение | колесо мыши / `+` `−` / перетаскивание, `ivc fit` |
| Масштаб по клавише: 100%, по высоте, по ширине, вписать | клавиши `1`–`6`, `ivc fit` |
| Навигация по папке | `→` `←` `Home` `End`, `ivc nav`, `ivc list` |
| Номер изображения, количество, сведения о файле, гистограмма | строка состояния, `I`, `H`, `ivc info`, `ivc histogram` |
| Настройки, темы, режимы | `F9`, `T`, `B`, `P`, `ivc settings` |
| Справка F1 | `F1`, `ImageViewer --help`, `ivc keys` |
| Тесты | `dotnet test` (раздел 1) |
| Графический интерфейс | Avalonia (Linux/Windows/macOS) и WPF (Windows), раздел 3 |